# 01b — Train GaitSet on CASIA-B with OpenGait (WS2, due ~Oct 24)

No CASIA-B GaitSet checkpoint is released by OpenGait, but He et al. attacked GaitSet, so we
need one to validate our He reproduction. We train it with OpenGait's own trainer and
`configs/opengait/gaitset_casiab_1gpu.yaml` (= OpenGait's gaitset.yaml with 1-GPU changes only).

**Resumable:** OpenGait saves every 2000 iterations. Each run trains one chunk (`CHUNK`
iterations, sized from the timing cell to stay < 3 h) and restarts from the latest checkpoint.
Copy `output/.../checkpoints` to the team Dataset / Drive after every chunk.
**Gate:** clean Rank-1 within ±1.0 pp of OpenGait's reported 95.8 / 90.0 / 75.4 (checked in Notebook 01).

In [ ]:
# --- Paths per platform. Edit ONLY this cell. -------------------------------------------
import os, sys, subprocess
from pathlib import Path

PLATFORM = "kaggle" if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") else \
           "colab" if "google.colab" in sys.modules else "local"
PATHS = {
    # Kaggle: team repo cloned into /kaggle/working; data + checkpoints from the private team Dataset.
    "kaggle": dict(repo="/kaggle/working/cross_attack_gait",
                   opengait="/kaggle/working/OpenGait",
                   dataset_root="/kaggle/input/casia-b-opengait/CASIA-B-pkl",
                   ckpt_dir="/kaggle/input/casia-b-opengait/checkpoints"),
    # Colab: Drive mirror of the same Dataset.
    "colab": dict(repo="/content/cross_attack_gait",
                  opengait="/content/OpenGait",
                  dataset_root="/content/drive/MyDrive/gait/CASIA-B-pkl",
                  ckpt_dir="/content/drive/MyDrive/gait/checkpoints"),
    # Local: set CASIAB_PKL and GAIT_CKPT_DIR (e.g. ../data/CASIA-B-pkl). CPU-only machines cannot run
    # OpenGait models (BaseModel hardcodes CUDA); official gate runs on GPU.
    "local": dict(repo=str(Path.cwd().parent), opengait=str(Path.cwd().parent.parent / "OpenGait"),
                  dataset_root=os.environ.get("CASIAB_PKL", ""), ckpt_dir=os.environ.get("GAIT_CKPT_DIR", "")),
}[PLATFORM]
TEAM_REPO_URL = "https://github.com/haragam22/gait-paper.git"   # e.g. https://github.com/<org>/cross_attack_gait.git (private; use a token on Kaggle)

if PLATFORM == "colab":
    from google.colab import drive; drive.mount("/content/drive")
if PLATFORM != "local" and not Path(PATHS["repo"]).exists():
    env = dict(os.environ)
    if PLATFORM == "kaggle":  # private repo: add a GitHub token as Kaggle secret GITHUB_TOKEN (Add-ons > Secrets)
        import base64
        from kaggle_secrets import UserSecretsClient
        auth = base64.b64encode(f"x-access-token:{UserSecretsClient().get_secret('GITHUB_TOKEN')}".encode()).decode()
        # token via env-only git config: never in argv, error messages, or .git/config (saved with notebook output)
        env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraheader",
                   GIT_CONFIG_VALUE_0=f"Authorization: Basic {auth}")
    subprocess.run(["git", "clone", TEAM_REPO_URL, PATHS["repo"]], check=True, env=env)
sys.path.insert(0, PATHS["repo"]); os.chdir(PATHS["repo"])

import src.utils as U
BASE = U.load_yaml("configs/baseline.yaml")
if not Path(PATHS["opengait"]).exists():
    subprocess.run(["git", "clone", "https://github.com/ShiqiYu/OpenGait.git", PATHS["opengait"]], check=True)
subprocess.run(["git", "-C", PATHS["opengait"], "checkout", "-q", BASE["opengait_commit"]], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "einops", "tensorboard"], check=True)
# Kaggle data: attach Notebook 01's saved output (it contains the packed CASIA-B-pkl) or the team Dataset.
if PLATFORM == "kaggle" and not Path(PATHS["dataset_root"]).exists():
    PATHS["dataset_root"] = str(next(p for pat in ("*/", "*/*/*/") for p in Path("/kaggle/input").glob(pat + "CASIA-B-pkl")))
print("dataset_root:", PATHS["dataset_root"])
BASE["opengait_dir"] = PATHS["opengait"]
BASE["dataset_root"] = PATHS["dataset_root"]

In [ ]:
SEED = BASE["seed"]
ENV = U.env_record(SEED, BASE["opengait_dir"], {"notebook": "01b_gaitset_training"})
ENV

In [ ]:
import re, shutil, yaml
CFG_SRC = Path(PATHS["repo"]) / "configs/opengait/gaitset_casiab_1gpu.yaml"
CKPT_DIR = Path(PATHS["opengait"]) / "output/CASIA-B/GaitSet/GaitSet/checkpoints"
PERSIST = Path(PATHS["ckpt_dir"]) / "gaitset_training"      # Drive / previous Kaggle version output
CKPT_DIR.mkdir(parents=True, exist_ok=True)
for p in sorted(PERSIST.glob("GaitSet-*.pt")) if PERSIST.exists() else []:   # restore earlier chunks
    if not (CKPT_DIR / p.name).exists():
        shutil.copy(p, CKPT_DIR / p.name)
done = max([int(re.search(r"-(\d+)\.pt", p.name).group(1)) for p in CKPT_DIR.glob("GaitSet-*.pt")], default=0)
print("latest saved iteration:", done)

In [ ]:
CHUNK = 2000          # first run: 2000 to time it; then set from the measured it/s (multiple of 2000, < 3 h)
TOTAL = 40000
target = min(done + CHUNK, TOTAL)
cfg = yaml.safe_load(open(CFG_SRC))
cfg["data_cfg"]["dataset_root"] = PATHS["dataset_root"]
cfg["trainer_cfg"]["total_iter"] = target
run_cfg = Path(PATHS["opengait"]) / "gaitset_casiab_run.yaml"
yaml.safe_dump(cfg, open(run_cfg, "w"))
cmd = ["torchrun", "--nproc_per_node=1", "opengait/main.py", "--cfgs", str(run_cfg), "--phase", "train",
       "--log_to_file"] + (["--iter", str(done)] if done else [])
print(" ".join(cmd))
import time; t0 = time.time()
subprocess.run(cmd, cwd=PATHS["opengait"], env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"}, check=True)
print(f"{target - done} iterations in {(time.time() - t0) / 60:.1f} min")

In [ ]:
# Persist after every chunk (Colab: Drive; Kaggle: this version's output -> add as input next time).
dest = PERSIST if PLATFORM == "colab" else Path("/kaggle/working/gaitset_training")
dest.mkdir(parents=True, exist_ok=True)
for p in CKPT_DIR.glob("GaitSet-*.pt"):
    shutil.copy(p, dest / p.name)
print(sorted(p.name for p in dest.glob("*.pt")))

## Findings, limitations, next step
- Time per 1000 iterations:
- When GaitSet-40000.pt exists: copy it to the team Dataset `checkpoints/`, rerun Notebook 01's GaitSet cell.